# ML Prediction for the upcoming FIFA world cup matches

## CELL 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
import os
import random
warnings.filterwarnings("ignore")
 
from sklearn.ensemble        import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.metrics         import classification_report
from sklearn.pipeline        import Pipeline
 
os.makedirs("../assets", exist_ok=True)
os.makedirs("../models", exist_ok=True)
 
plt.rcParams.update({
    "figure.facecolor" : "#0d1117", "axes.facecolor"  : "#161b22",
    "axes.edgecolor"   : "#30363d", "axes.labelcolor" : "#c9d1d9",
    "xtick.color"      : "#8b949e", "ytick.color"     : "#8b949e",
    "text.color"       : "#c9d1d9", "grid.color"      : "#21262d",
    "grid.linestyle"   : "--",      "grid.alpha"      : 0.5,
    "font.family"      : "sans-serif",
    "axes.titlesize"   : 13,        "axes.titleweight" : "bold",
    "axes.titlepad"    : 12,
})
GOLD="#FFD700"; GREEN="#238636"; RED="#da3633"
BLUE="#58a6ff"; ORANGE="#f0883e"; PURPLE="#bc8cff"; GREY="#8b949e"
CONF_COLORS = {"UEFA":BLUE,"CONMEBOL":GOLD,"CAF":GREEN,
               "AFC":ORANGE,"CONCACAF":PURPLE,"OFC":GREY}
 
print("Imports completed successfully.")

## CELL 2: Load cleaned datasets

In [ ]:
results     = pd.read_csv("../data/cleaned/results_clean.csv")
wc_finals   = pd.read_csv("../data/cleaned/wc_finals_clean.csv")
wc_top4     = pd.read_csv("../data/cleaned/wc_top4_clean.csv")
rankings    = pd.read_csv("../data/cleaned/rankings_clean.csv")
pre_wc      = pd.read_csv("../data/cleaned/pre_wc_rankings.csv")
 
results["date"]       = pd.to_datetime(results["date"])
rankings["rank_date"] = pd.to_datetime(rankings["rank_date"])
pre_wc["rank_date"]   = pd.to_datetime(pre_wc["rank_date"])
 
for df in [wc_finals, wc_top4]:
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].replace("West Germany", "Germany")
 
print(f"results    : {results.shape}")
print(f"rankings   : {rankings.shape}")

# SECTION A — TEAM STRENGTH PROFILES
##  Pre-compute a strength profile for every team

## CELL 3: Latest FIFA rankings

In [ ]:
latest_date    = rankings["rank_date"].max()
latest_rankings = rankings[rankings["rank_date"] == latest_date].copy()
print(f"Using rankings from: {latest_date.date()}")
print(f"Teams ranked: {len(latest_rankings)}")

## CELL 4: Recent form calculator 
### Win rate, goals for/against in last 24 months

In [56]:
def get_team_form(team, before_date=None, months=24):
    if before_date is None:
        before_date = pd.Timestamp("2026-06-01")
    cutoff = before_date - pd.DateOffset(months=months)
 
    mask = (
        ((results["home_team"] == team) | (results["away_team"] == team)) &
        (results["date"] >= cutoff) & (results["date"] < before_date)
    )
    recent = results[mask].copy()
    if len(recent) == 0:
        return {"win_rate": 0.45, "goals_for": 1.2, "goals_against": 1.2}
 
    wins = 0; gf = 0; ga = 0
    for _, r in recent.iterrows():
        if r["home_team"] == team:
            f, a = r["home_score"], r["away_score"]
        else:
            f, a = r["away_score"], r["home_score"]
        gf += f; ga += a
        if f > a: wins += 1
 
    n = len(recent)
    return {"win_rate": round(wins/n, 3),
            "goals_for": round(gf/n, 2),
            "goals_against": round(ga/n, 2)}

## CELL 5: Build team profiles for all WC 2026 teams 
### Official WC 2026 Group Draw (December 5, 2025)
### Playoff spots filled with most likely qualifier


In [ ]:
# Copy and paste this into a cell, then press Shift + Enter
# ...existing code...
world_cup_teams = [
    "Algeria", "Argentina", "Australia", "Austria", "Belgium", "Bosnia and Herzegovina",
    "Brazil", "Cape Verde", "Canada", "Colombia", "Congo DR", "Ivory Coast",
    "Croatia", "Curacao", "Czechia", "Ecuador", "Egypt", "England", "France",
    "Germany", "Ghana", "Haiti", "Iran", "Iraq", "Japan", "Jordan", "South Korea",
    "Mexico", "Morocco", "Netherlands", "New Zealand", "Norway", "Panama", "Paraguay",
    "Portugal", "Qatar", "Saudi Arabia", "Scotland", "Senegal", "South Africa",
    "Spain", "Sweden", "Switzerland", "Tunisia", "Turkiye", "United States",
    "Uruguay", "Uzbekistan"
]
# ...existing code...

# Ensure canonical names used in WC2026_GROUPS exist in the master list
if "Denmark" not in world_cup_teams:
    world_cup_teams.append("Denmark")
if "Italy" not in world_cup_teams:
    world_cup_teams.append("Italy")
# ...existing code...

# Print the total count to verify
print(f"Total teams loaded: {len(world_cup_teams)}")
WC2026_GROUPS = {
    "A": ["Mexico",        "South Africa",  "South Korea",  "Denmark"],
    "B": ["Canada",        "Italy",         "Qatar",        "Switzerland"],
    "C": ["Brazil",        "Morocco",       "Haiti",        "Scotland"],
    "D": ["United States", "Paraguay",      "Australia",    "Turkiye"],
    "E": ["Germany",       "Curacao",       "Ivory Coast",  "Ecuador"],
    "F": ["Netherlands",   "Japan",         "Sweden",      "Tunisia"],
    "G": ["Belgium",       "Egypt",         "Iran",         "New Zealand"],
    "H": ["Spain",         "Cape Verde",    "Saudi Arabia", "Uruguay"],
    "I": ["France",        "Senegal",       "Iraq",         "Norway"],
    "J": ["Argentina",     "Algeria",       "Austria",      "Jordan"],
    "K": ["Portugal",      "Congo DR",       "Uzbekistan",   "Colombia"],
    "L": ["England",       "Croatia",       "Ghana",        "Panama"],
}

CONFEDERATION_MAP = {
    "Mexico":"CONCACAF","South Africa":"CAF","South Korea":"AFC","Denmark":"UEFA",
    "Canada":"CONCACAF","Italy":"UEFA","Qatar":"AFC","Switzerland":"UEFA",
    "Brazil":"CONMEBOL","Morocco":"CAF","Haiti":"CONCACAF","Scotland":"UEFA",
    "United States":"CONCACAF","Paraguay":"CONMEBOL","Australia":"AFC","Turkiye":"UEFA",
    "Germany":"UEFA","Curacao":"CONCACAF","Ivory Coast":"CAF","Ecuador":"CONMEBOL",
    "Netherlands":"UEFA","Japan":"AFC","Ukraine":"UEFA","Tunisia":"CAF",
    "Belgium":"UEFA","Egypt":"CAF","Iran":"AFC","New Zealand":"OFC",
    "Spain":"UEFA","Cape Verde":"CAF","Saudi Arabia":"AFC","Uruguay":"CONMEBOL",
    "France":"UEFA","Senegal":"CAF","Iraq":"AFC","Norway":"UEFA",
    "Argentina":"CONMEBOL","Algeria":"CAF","Austria":"UEFA","Jordan":"AFC",
    "Portugal":"UEFA","Jamaica":"CONCACAF","Uzbekistan":"AFC","Colombia":"CONMEBOL",
    "England":"UEFA","Croatia":"UEFA","Ghana":"CAF","Panama":"CONCACAF",
}
 
HOST_TEAMS = {"United States", "Mexico", "Canada"}

# Build profile for each team
print("Building team profiles...")
world_cup_teams_list = {}
for team in world_cup_teams:
    rank_row = latest_rankings[latest_rankings["country_full"] == team]
    rank   = int(rank_row["rank"].values[0])   if len(rank_row) > 0 else 55
    points = float(rank_row["total_points"].values[0]) if len(rank_row) > 0 else 1000.0
 
    top4_row   = wc_top4[wc_top4["team"] == team]
    titles_count    = int(top4_row["titles_count"].values[0])     if len(top4_row) > 0 else 0
    top4_count = int(top4_row["top4_total"].values[0]) if len(top4_row) > 0 else 0
 
    form = get_team_form(team)
 
    world_cup_teams_list[team] = {
        "rank"          : rank,
        "points"        : points,
        "titles_count"  : titles_count,
        "top4_count"    : top4_count,
        "win_rate"      : form["win_rate"],
        "goals_for"     : form["goals_for"],
        "goals_against" : form["goals_against"],
        "confederation" : CONFEDERATION_MAP.get(team, "UEFA"),
        "is_host"       : 1 if team in HOST_TEAMS else 0,
    }
 
print("Team profiles built successfully.")
print(f"Sample — Argentina: {world_cup_teams_list.get('Argentina', 'Not found')}")

In [113]:
# ...existing code...
import unicodedata

def _norm(s):
    return ''.join(
        c for c in unicodedata.normalize("NFKD", str(s))
        if not unicodedata.combining(c)
    ).lower().replace(" ", "").replace("'", "").replace("-", "")

# Known aliases: normalized_alias -> canonical name in world_cup_teams
ALIASES = {
    "southkorea": "Korea Republic",
    "capeverde": "Cabo Verde",
    "ivorycoast": "Côte d'Ivoire",
    "curacao": "Curaçao",
    "turkiye": "Türkiye",
    "saintkitts": "Saint Kitts and Nevis",  # example if needed
}

# build normalized lookup for exact canonical matches
team_lookup = {_norm(t): t for t in world_cup_teams}

def find_team_key(name):
    k = _norm(name)
    if k in team_lookup:
        return team_lookup[k]
    if k in ALIASES:
        return ALIASES[k]
    raise KeyError(f"Team name not found (use canonical name or add alias): {name}")

# Normalize WC2026_GROUPS names once, immediately after groups are defined
if "Italy" not in world_cup_teams:
    world_cup_teams.append("Italy")
WC2026_GROUPS = {g: [find_team_key(t) for t in teams] for g, teams in WC2026_GROUPS.items()}
# ...existing code...

##  SECTION B — MATCH PREDICTION MODEL
###  Train on 43,000 historical matches to predict W/D/L

## CELL 6: Build match-level training dataset

In [ ]:
print("Building match training dataset")
 
# Use last 20 years of competitive matches — more relevant than 1872 data
TRAIN_CUTOFF = pd.Timestamp("2006-01-01")
IMPORTANT_TOURNAMENTS = [
    "FIFA World Cup", "FIFA World Cup qualification",
    "Copa América", "UEFA Euro", "UEFA Euro qualification",
    "UEFA Nations League", "CONMEBOL–UEFA Cup of Champions",
    "African Cup of Nations", "AFC Asian Cup",
    "CONCACAF Gold Cup", "Friendly",
]
 
match_data = results[
    (results["date"] >= TRAIN_CUTOFF) &
    (results["tournament"].isin(IMPORTANT_TOURNAMENTS))
].copy()
 
print(f"Matches for training: {len(match_data):,}")
 
# Build features per match
match_rows = []
for _, row in match_data.iterrows():
    home = row["home_team"]
    away = row["away_team"]
 
    # Get rankings closest to match date
    before = row["date"]
    rank_slice = rankings[rankings["rank_date"] <= before]
    if len(rank_slice) == 0:
        continue
    latest_before = rank_slice["rank_date"].max()
    snap = rank_slice[rank_slice["rank_date"] == latest_before]
 
    h_rank_row = snap[snap["country_full"] == home]
    a_rank_row = snap[snap["country_full"] == away]
 
    h_rank = int(h_rank_row["rank"].values[0])   if len(h_rank_row) > 0 else 50
    a_rank = int(a_rank_row["rank"].values[0])   if len(a_rank_row) > 0 else 50
    h_pts  = float(h_rank_row["total_points"].values[0]) if len(h_rank_row) > 0 else 1000.0
    a_pts  = float(a_rank_row["total_points"].values[0]) if len(a_rank_row) > 0 else 1000.0
 
    # Tournament importance weight
    is_wc       = 1 if row["tournament"] == "FIFA World Cup" else 0
    is_friendly = 1 if row["tournament"] == "Friendly" else 0
 
    # Result
    if row["home_score"] > row["away_score"]:   result = 2  # home win
    elif row["home_score"] == row["away_score"]: result = 1  # draw
    else:                                         result = 0  # away win
 
    match_rows.append({
        "rank_diff"    : a_rank - h_rank,   # positive = home team better ranked
        "pts_diff"     : h_pts  - a_pts,    # positive = home team more points
        "h_rank"       : h_rank,
        "a_rank"       : a_rank,
        "is_wc"        : is_wc,
        "is_friendly"  : is_friendly,
        "result"       : result,
    })
 
match_df = pd.DataFrame(match_rows)
print(f"Match training rows: {len(match_df):,}")
print(f"Result distribution: {match_df['result'].value_counts().to_dict()}")

## CELL 7: Train match model

In [ ]:
MATCH_FEATURES = ["rank_diff", "pts_diff", "h_rank", "a_rank",
                  "is_wc", "is_friendly"]
 
X_match = match_df[MATCH_FEATURES].fillna(0)
y_match = match_df["result"]  # 0=away win, 1=draw, 2=home win
 
# StratifiedKFold to preserve W/D/L proportions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
models = {
    "Logistic Regression"  : Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=500, random_state=42))]), # Removed multi_class
    "Random Forest"        : RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting" : HistGradientBoostingClassifier(
        max_iter=200, max_depth=5, min_samples_leaf=8,
        class_weight="balanced", random_state=42
    ),
}

 
print("=== Match Model — 5-Fold Cross Validation ===\n")
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_match, y_match, cv=cv, scoring="f1_macro")
    cv_results[name] = scores
    print(f"{name:<28} F1-macro: {scores.mean():.3f} (+/- {scores.std():.3f})")

## CELL 8: Train final match model
### Using HistGradientBoosting as the main model

In [ ]:
match_model = HistGradientBoostingClassifier(
    max_iter=300, max_depth=5, min_samples_leaf=8,
    class_weight="balanced", random_state=42
)
match_model.fit(X_match, y_match)
 
print("Match model trained ✓")
print(classification_report(y_match, match_model.predict(X_match),
      target_names=["Away Win", "Draw", "Home Win"]))
 
joblib.dump(match_model, "../models/match_predictor.pkl")

## CELL 9: Model comparison chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = [BLUE, GOLD, GREEN]
 
bars = ax.bar(names, means, color=colors, edgecolor="none",
              width=0.5, yerr=stds, capsize=6,
              error_kw={"color":GREY,"linewidth":1.5})
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{m:.3f}", ha="center", va="bottom",
            fontsize=12, fontweight="bold", color="#c9d1d9")
ax.set_ylim(0, 0.6)
ax.set_ylabel("F1-Macro Score (higher = better)")
ax.set_title("Match Prediction Model Comparison — 5-Fold CV")
ax.grid(axis="y"); ax.set_frame_on(False)
plt.tight_layout()
plt.savefig("../assets/09_model_comparison.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()


#  SECTION C — TOURNAMENT SIMULATOR

## CELL 10: Match probability function

In [63]:
def get_match_probabilities(team_a, team_b, neutral=True):
    """
    Returns (prob_a_wins, prob_draw, prob_b_wins) for a match.
    team_a is treated as the 'home' team in the feature vector.
    neutral=True reduces the home advantage effect.
    """
    pa = world_cup_teams_list.get(team_a, {})
    pb = world_cup_teams_list.get(team_b, {})
 
    rank_diff = pb.get("rank", 50) - pa.get("rank", 50)
    pts_diff  = pa.get("points", 1000) - pb.get("points", 1000)
 
    # Slight host bonus for USA/Canada/Mexico games
    if not neutral and team_a in HOST_TEAMS:
        rank_diff += 5
        pts_diff  += 50
 
    features = pd.DataFrame([{
        "rank_diff"   : rank_diff,
        "pts_diff"    : pts_diff,
        "h_rank"      : pa.get("rank", 50),
        "a_rank"      : pb.get("rank", 50),
        "is_wc"       : 1,
        "is_friendly" : 0,
    }])
 
    probs = match_model.predict_proba(features)[0]
    # probs[0]=away win, probs[1]=draw, probs[2]=home win
    return probs[2], probs[1], probs[0]  # (team_a_wins, draw, team_b_wins)
 
# Quick test
p = get_match_probabilities("Argentina", "France")
print(f"Argentina vs France → Argentina:{p[0]:.2%} Draw:{p[1]:.2%} France:{p[2]:.2%}")

Argentina vs France → Argentina:29.57% Draw:29.78% France:40.65%


## CELL 11: Simulate one group stage match

In [98]:
def simulate_match(team_a, team_b, knockout=False):
    """
    Simulates one match. Returns (winner, loser) or
    (team_a_goals, team_b_goals) for group stage.
    In knockout: draws go to extra time (50/50 with strength bias).
    """
    pa_win, draw, pb_win = get_match_probabilities(team_a, team_b)
    rand = random.random()
 
    if rand < pa_win:
        return (team_a, team_b, "A")   # team_a wins
    elif rand < pa_win + draw:
        if knockout:
            # Extra time: bias toward better-ranked team
            ra = world_cup_teams_list[team_a]["rank"]
            rb = world_cup_teams_list[team_b]["rank"]
            p_a_et = rb / (ra + rb)   # lower rank number = better
            winner = team_a if random.random() < p_a_et else team_b
            loser  = team_b if winner == team_a else team_a
            return (winner, loser, "ET")
        return (team_a, team_b, "D")   # draw (group stage)
    else:
        return (team_b, team_a, "B")   # team_b wins

## CELL 12: Group stage simulation

In [73]:
def simulate_group(group_teams):
    """
    Runs round-robin for one group (each team plays 3 matches).
    Returns standings: list of (team, points, gd, gf) sorted.
    """
    standings = {t: {"pts":0, "gd":0, "gf":0} for t in group_teams}
 
    # Generate all matchups (6 per group)
    matchups = [(a, b) for i, a in enumerate(group_teams)
                       for b in group_teams[i+1:]]
 
    for team_a, team_b in matchups:
        winner, loser, outcome = simulate_match(team_a, team_b)
 
        # Simulate approximate goals for goal difference
        ra = world_cup_teams_list[team_a]["rank"]
        rb = world_cup_teams_list[team_b]["rank"]
        avg_gf_a = world_cup_teams_list[team_a]["goals_for"]
        avg_gf_b = world_cup_teams_list[team_b]["goals_for"]
 
        goals_a = max(0, int(np.random.poisson(avg_gf_a)))
        goals_b = max(0, int(np.random.poisson(avg_gf_b)))
 
        if outcome == "A":   # team_a won
            goals_a = max(goals_a, goals_b + 1)
            standings[team_a]["pts"] += 3
        elif outcome == "B": # team_b won
            goals_b = max(goals_b, goals_a + 1)
            standings[team_b]["pts"] += 3
        else:                # draw
            goals_a = goals_b = min(goals_a, goals_b)
            standings[team_a]["pts"] += 1
            standings[team_b]["pts"] += 1
 
        standings[team_a]["gd"] += goals_a - goals_b
        standings[team_b]["gd"] += goals_b - goals_a
        standings[team_a]["gf"] += goals_a
        standings[team_b]["gf"] += goals_b
 
    # Sort: points → goal difference → goals scored
    sorted_teams = sorted(standings.items(),
                          key=lambda x: (x[1]["pts"], x[1]["gd"], x[1]["gf"]),
                          reverse=True)
    return sorted_teams  # [(team, {pts,gd,gf}), ...]

## CELL 13: Full tournament simulation

In [66]:
def simulate_tournament():
    """
    Simulates the full WC 2026 tournament once.
    Returns the winner.
    WC 2026 format: 12 groups → top 2 each = 24 + 8 best 3rd = 32 teams
    """
    # ── GROUP STAGE ──
    all_group_results = {}
    third_place_teams = []
 
    for group, teams in WC2026_GROUPS.items():
        standings = simulate_group(teams)
        all_group_results[group] = standings
        third_place_teams.append((standings[2][0], standings[2][1]))
 
    # Select 8 best 3rd-place teams (by points, then gd, then gf)
    third_place_teams.sort(
        key=lambda x: (x[1]["pts"], x[1]["gd"], x[1]["gf"]),
        reverse=True
    )
    best_thirds = [t[0] for t in third_place_teams[:8]]
 
    # Assemble Round of 32 (32 teams)
    round32 = []
    for group, standings in all_group_results.items():
        round32.append(standings[0][0])  # group winner
        round32.append(standings[1][0])  # runner-up
 
    # Add 8 best 3rd-place teams
    round32 += best_thirds
    random.shuffle(round32)  # randomise bracket
 
    # ── KNOCKOUT ROUNDS ──
    def play_knockout_round(teams):
        """Pairs teams and plays one knockout round."""
        winners = []
        for i in range(0, len(teams), 2):
            if i+1 >= len(teams):
                winners.append(teams[i])  # bye
                continue
            winner, _, _ = simulate_match(teams[i], teams[i+1], knockout=True)
            winners.append(winner)
        return winners
 
    r32   = play_knockout_round(round32)    # 32 → 16
    r16   = play_knockout_round(r32)        # 16 → 8
    qf    = play_knockout_round(r16)        #  8 → 4
    sf    = play_knockout_round(qf)         #  4 → 2
    final = play_knockout_round(sf)         #  2 → 1
 
    return final[0]  # Tournament winner

## CELL 14: Monte Carlo simulation

In [ ]:
N_SIMULATIONS = 10000
print(f"Running {N_SIMULATIONS:,} tournament simulations...")
print("This takes 1 hr to run.")
 
win_counts = {team: 0 for team in world_cup_teams}
 
for i in range(N_SIMULATIONS):
    winner = simulate_tournament()
    if winner in win_counts:
        win_counts[winner] += 1
    if (i+1) % 1000 == 0:
        print(f"  Completed {i+1:,} / {N_SIMULATIONS:,} simulations...")
 
print(f"\nSimulations complete ✓")

## CELL 15: Calculate win probabilities

In [ ]:
predictions = pd.DataFrame([
    {
        "team"          : team,
        "confederation" : CONFEDERATION_MAP.get(team, "UEFA"),
        "fifa_rank"     : world_cup_teams_list[team]["rank"],
        "wc_titles"     : world_cup_teams_list[team]["titles_count"],
        "wins_in_sim"   : count,
        "win_prob_pct"  : round(count / N_SIMULATIONS * 100, 2),
    }
    for team, count in win_counts.items()
]).sort_values("win_prob_pct", ascending=False).reset_index(drop=True)
 
predictions.index += 1
 
print("  WC 2026 WINNER PROBABILITY (Monte Carlo, 10,000 sims)")
print(predictions.head(20).to_string())

#  SECTION D — VISUALISATIONS

## CELL 16: Top 16 win probability chart

In [ ]:
top16 = predictions.head(16).copy()
top16 = top16.sort_values("win_prob_pct")
bar_colors = [CONF_COLORS.get(c, GREY) for c in top16["confederation"]]
 
fig, ax = plt.subplots(figsize=(11, 9))
bars = ax.barh(top16["team"], top16["win_prob_pct"],
               color=bar_colors, edgecolor="none", height=0.7)
 
for bar, val in zip(bars, top16["win_prob_pct"]):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            f"  {val:.1f}%", va="center", fontsize=10,
            fontweight="bold", color="#c9d1d9")
 
ax.set_xlabel("Win Probability — 10,000 Monte Carlo Simulations (%)")
ax.set_title("FIFA World Cup 2026 — Predicted Win Probabilities\n"
             "Match-by-Match Tournament Simulation", pad=14)
ax.set_xlim(0, top16["win_prob_pct"].max() * 1.28)
ax.grid(axis="x"); ax.set_frame_on(False)
 
patches = [mpatches.Patch(color=v, label=k)
           for k, v in CONF_COLORS.items()
           if k in top16["confederation"].values]
ax.legend(handles=patches, loc="lower right",
          facecolor="#161b22", edgecolor="#30363d",
          labelcolor="#c9d1d9", title="Confederation")
 
plt.tight_layout()
plt.savefig("../assets/12_wc2026_simulation_results.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()

## CELL 17: Confederation win probability

In [ ]:
conf_prob = (predictions.groupby("confederation")["win_prob_pct"]
             .sum().sort_values(ascending=False))
 
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(conf_prob.index, conf_prob.values,
              color=[CONF_COLORS.get(c, GREY) for c in conf_prob.index],
              edgecolor="none", width=0.6)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f"{bar.get_height():.1f}%", ha="center", va="bottom",
            fontsize=11, fontweight="bold", color="#c9d1d9")
ax.set_ylabel("Total Win Probability (%)")
ax.set_title("WC 2026 — Cumulative Win Probability by Confederation")
ax.grid(axis="y"); ax.set_frame_on(False)
plt.tight_layout()
plt.savefig("../assets/13_confederation_win_probability.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()

## CELL 18: Head-to-head probability matrix (top 8 teams)

In [ ]:
top8 = predictions.head(8)["team"].tolist()
matrix = pd.DataFrame(index=top8, columns=top8, dtype=float)
 
for team_a in top8:
    for team_b in top8:
        if team_a == team_b:
            matrix.loc[team_a, team_b] = np.nan
        else:
            pa, _, pb = get_match_probabilities(team_a, team_b)
            matrix.loc[team_a, team_b] = round(pa * 100, 1)
 
fig, ax = plt.subplots(figsize=(10, 8))
mask = pd.isna(matrix.astype(float))
sns.heatmap(matrix.astype(float), annot=True, fmt=".0f",
            cmap="RdYlGn", center=50, vmin=20, vmax=80,
            linewidths=0.5, linecolor="#0d1117",
            mask=mask, ax=ax,
            cbar_kws={"label": "Win % (row team vs column team)"})
ax.set_title("Head-to-Head Win Probabilities — Top 8 Contenders\n"
             "(Row team win % against column team)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../assets/14_head_to_head_matrix.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()

## CELL 19: Group-by-group breakdown

In [ ]:
group_probs = []
for group, teams in WC2026_GROUPS.items():
    for team in teams:
        prob = predictions[predictions["team"]==team]["win_prob_pct"].values
        group_probs.append({
            "group": group,
            "team": team,
            "win_prob": prob[0] if len(prob) > 0 else 0,
            "confederation": CONFEDERATION_MAP.get(team, "UEFA"),
            "rank": world_cup_teams_list[team]["rank"],
        })
 
group_df = pd.DataFrame(group_probs)
 
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()
 
for i, group in enumerate(sorted(WC2026_GROUPS.keys())):
    gdata = group_df[group_df["group"]==group].sort_values("win_prob", ascending=False)
    ax = axes[i]
    colors = [CONF_COLORS.get(c, GREY) for c in gdata["confederation"]]
    bars = ax.bar(gdata["team"], gdata["win_prob"], color=colors,
                  edgecolor="none", width=0.6)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f"{bar.get_height():.1f}%", ha="center", va="bottom",
                fontsize=8, color="#c9d1d9")
    ax.set_title(f"Group {group}", fontsize=11, fontweight="bold")
    ax.set_xticklabels(gdata["team"], rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Win %", fontsize=8)
    ax.set_frame_on(False)
    ax.grid(axis="y", alpha=0.4)
 
plt.suptitle("WC 2026 — Win Probability by Group",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../assets/15_group_breakdown.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()

## CELL 20: Save results

In [ ]:
predictions.to_csv("../data/cleaned/wc2026_simulation_results.csv",
                   index_label="predicted_rank")
joblib.dump(match_model, "../models/match_predictor.pkl")
 
print("Results saved")
print("Model saved")

## CELL 21: Final summary

In [ ]:
print()
print("  WC 2026 TOP 12 — MONTE CARLO SIMULATION RESULTS")
print(f"  Based on {N_SIMULATIONS:,} full tournament simulations")
for i, row in predictions.head(12).iterrows():
    bar = "█" * max(1, int(row["win_prob_pct"] * 1.5))
    print(f"  {i:2}. {row['team']:<22} {row['win_prob_pct']:5.1f}%  {bar}")
 
print(f"""
How the model works:
  1. Trained on {len(match_df):,} historical international matches (2006-2023)
  2. Predicts W/D/L probability for every match-up
  3. Simulates full WC 2026 bracket {N_SIMULATIONS:,} times
  4. Counts how often each team lifts the trophy
 
Outputs:
  data/cleaned/wc2026_simulation_results.csv
  models/match_predictor.pkl
  assets/12 through assets/15 (4 charts)
 
Add 'models/' to your .gitignore before pushing.
""")